# Chapitre 14 - Cours 2 : classification (analyse de groupements)

Ce notebook couvre la section 14.3 du livre (*cluster analysis*) :

1. matrices de proximité et dissimilarités entre objets ;
2. algorithmes combinatoires, **K-means** et lien avec les mélanges gaussiens ;
3. K-means sur les données de **micro-puces NCI** et quantification vectorielle ;
4. **K-médoïdes** sur les dissimilarités entre pays (figure 14.10) ;
5. choix du nombre de groupes et **statistique Gap** (figure 14.11) ;
6. **classification hiérarchique** et dendrogrammes sur les données NCI
   (figures 14.12 à 14.14).

Le mot anglais *clustering* sera traduit par « classification » (non supervisée) ou
« partitionnement » ; un *cluster* est un « groupe » ou une « classe ».

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from math import factorial
from scipy.special import comb
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS, decision_grid
from esl import datasets
from esl import ch14_non_supervise as ns
set_style()

## 1. Objectifs de la classification

On veut regrouper $N$ objets en sous-ensembles (groupes) tels que les objets d'un même groupe
soient plus « proches » entre eux que d'objets de groupes différents. On peut aussi vouloir
organiser les groupes en **hiérarchie** (des groupes emboîtés dans des groupes).

Toute méthode repose sur une notion de **dissimilarité** entre objets. Comme le choix d'une
fonction de perte en apprentissage supervisé, ce choix vient de considérations extérieures
aux données (connaissance du domaine) et il est souvent plus important que l'algorithme
lui-même.

## 2. Matrices de proximité

Les données se présentent parfois directement sous forme d'une matrice $N \times N$
$\mathbf{D} = (d_{ii'})$ de dissimilarités : par exemple des jugements moyens d'étudiants sur la
ressemblance politique de pays (section 14.3.1). La plupart des algorithmes supposent

* $d_{ii'} \ge 0$ et $d_{ii} = 0$ ;
* $\mathbf{D}$ symétrique (sinon on la remplace par $(\mathbf{D} + \mathbf{D}^T)/2$).

Des dissimilarités subjectives ne vérifient pas forcément l'**inégalité triangulaire**
$d_{ii'} \le d_{ik} + d_{ki'}$ : ce ne sont pas toujours des distances au sens mathématique.

In [2]:
pays = datasets.load_countries()
Dp = pays.to_numpy()
noms_pays = list(pays.index)
print("Symétrique :", np.allclose(Dp, Dp.T), "; diagonale nulle :", np.all(np.diag(Dp) == 0))
viol = sum(Dp[i, j] > Dp[i, k] + Dp[k, j] + 1e-12
           for i in range(12) for j in range(12) for k in range(12))
print("Violations de l'inégalité triangulaire :", viol)
pays.round(2)

Symétrique : True ; diagonale nulle : True
Violations de l'inégalité triangulaire : 0


,BEL,BRA,CHI,CUB,EGY,FRA,IND,ISR,USA,USS,YUG,ZAI
BEL,0.00,5.58,7.00,7.08,4.83,2.17,6.42,3.42,2.50,6.08,5.25,4.75
BRA,5.58,0.00,6.50,7.00,5.08,5.75,5.00,5.50,4.92,6.67,6.83,3.00
CHI,7.00,6.50,0.00,3.83,8.17,6.67,5.58,6.42,6.25,4.25,4.50,6.08
CUB,7.08,7.00,3.83,0.00,5.83,6.92,6.00,6.42,7.33,2.67,3.75,6.67
EGY,4.83,5.08,8.17,5.83,0.00,4.92,4.67,5.00,4.50,6.00,5.75,5.00
FRA,2.17,5.75,6.67,6.92,4.92,0.00,6.42,3.92,2.25,6.17,5.42,5.58
IND,6.42,5.00,5.58,6.00,4.67,6.42,0.00,6.17,6.33,6.17,6.08,4.83
ISR,3.42,5.50,6.42,6.42,5.00,3.92,6.17,0.00,2.75,6.92,5.83,6.17
USA,2.50,4.92,6.25,7.33,4.50,2.25,6.33,2.75,0.00,6.17,6.67,5.67
USS,6.08,6.67,4.25,2.67,6.00,6.17,6.17,6.92,6.17,0.00,3.67,6.50


Ici (tableau 14.3 du livre), l'inégalité triangulaire est respectée, mais rien ne
l'imposait.

## 3. Dissimilarités construites à partir des variables

Le plus souvent on dispose de mesures $x_{ij}$ ($i = 1..N$ objets, $j = 1..p$ variables). On
définit une dissimilarité $d_j$ par variable et on les additionne (éq. (14.20)) :

$$D(x_i, x_{i'}) = \sum_{j=1}^p d_j(x_{ij}, x_{i'j}).$$

Le choix le plus courant est le carré de l'écart, $d_j(x_{ij}, x_{i'j}) = (x_{ij} -
x_{i'j})^2$ (14.21), ce qui donne le carré de la distance euclidienne.

Selon le **type** de variable :

* **quantitative** : $(x_{ij} - x_{i'j})^2$ ou $|x_{ij} - x_{i'j}|$ ;
* **ordinale** à $M$ niveaux : on remplace le niveau $i$ par le score $\frac{i - 1/2}{M}$
  (éq. (14.23)) puis on la traite comme quantitative ;
* **catégorielle** à $M$ modalités : on se donne une matrice $M \times M$ de pertes $L_{rr'}$
  symétrique, nulle sur la diagonale ; le choix le plus simple est $L_{rr'} = 1$ si $r \ne r'$.

### Corrélation et distance euclidienne

On utilise parfois la **corrélation** entre deux objets (calculée sur leurs $p$ variables),
éq. (14.22) :

$$\rho(x_i, x_{i'}) = \frac{\sum_j (x_{ij} - \bar x_i)(x_{i'j} - \bar x_{i'})}
{\sqrt{\sum_j (x_{ij} - \bar x_i)^2 \sum_j (x_{i'j} - \bar x_{i'})^2}},
\qquad \bar x_i = \frac1p \sum_j x_{ij}.$$

**Propriété.** Si chaque objet est **standardisé** ($\bar x_i = 0$ et $\sum_j x_{ij}^2 = 1$), alors
$\sum_j (x_{ij} - x_{i'j})^2 = 2\,(1 - \rho(x_i, x_{i'}))$.

*Preuve.* On développe le carré :
$\sum_j (x_{ij} - x_{i'j})^2 = \sum_j x_{ij}^2 + \sum_j x_{i'j}^2 - 2 \sum_j x_{ij} x_{i'j}$.
Les deux premiers termes valent 1 par standardisation, et comme les moyennes sont nulles et
les normes égales à 1, $\rho(x_i, x_{i'}) = \sum_j x_{ij} x_{i'j}$. D'où $2 - 2\rho$. $\square$

Classer selon la corrélation revient donc à classer selon la distance euclidienne d'objets
standardisés. Vérification numérique :

In [ ]:
rng = np.random.RandomState(0)
A = rng.normal(size=(2, 30))
As = (A - A.mean(1, keepdims=True))
As /= np.sqrt((As ** 2).sum(1, keepdims=True))
print("somme des carrés :", ((As[0] - As[1]) ** 2).sum())
print("2 (1 - rho)      :", 2 * (1 - np.corrcoef(A)[0, 1]))

## 4. Pondération des variables : le piège de la standardisation

On peut pondérer les variables (éq. (14.24)) : $D(x_i, x_{i'}) = \sum_j w_j\, d_j(x_{ij},
x_{i'j})$ avec $\sum_j w_j = 1$. Mais l'**influence** d'une variable ne dépend pas que de son
poids : elle dépend aussi de la dissimilarité moyenne

$$\bar d_j = \frac{1}{N^2} \sum_{i=1}^N \sum_{i'=1}^N d_j(x_{ij}, x_{i'j}) \qquad (14.25).$$

L'influence relative de la variable $j$ sur la dissimilarité moyenne entre objets est
$w_j \bar d_j$.

**Calcul de $\bar d_j$ pour le carré de l'écart.** Notons $\bar x_j$ la moyenne de la colonne
$j$. On écrit $x_{ij} - x_{i'j} = (x_{ij} - \bar x_j) - (x_{i'j} - \bar x_j)$ et on développe :

$$\frac{1}{N^2}\sum_{i,i'} (x_{ij} - x_{i'j})^2
  = \frac{1}{N}\sum_i (x_{ij} - \bar x_j)^2 + \frac{1}{N}\sum_{i'} (x_{i'j} - \bar x_j)^2
  - \frac{2}{N^2} \Big(\sum_i (x_{ij} - \bar x_j)\Big)\Big(\sum_{i'} (x_{i'j} - \bar x_j)\Big).$$

Le dernier produit est nul (somme des écarts à la moyenne), d'où
$\bar d_j = 2\,\mathrm{var}_j$ (éq. (14.27)), avec $\mathrm{var}_j$ la variance empirique
(normalisée par $N$).